# DEP Mapping Conversion

**Source File:** `DEP`
**Conversion Timestamp:** 2024-07-30T12:00:00Z

This notebook migrates the ODI `DEP` mapping, converting source data from `DEPARTMENTS` to `TRG_DEP` within the `test` schema. It implements a standard upsert (MERGE) pattern for data integration, including staging, flow processing, and error handling components.

In [ ]:
dbutils.widgets.text("V_ETL_JOB_TYPE", "F", "1. ETL Job Type (F=Full, I=Incremental)")
dbutils.widgets.text("V_DATASOURCE_NUM_ID", "1", "2. Data Source Number ID")
dbutils.widgets.text("V_ETL_PROC_WID", "1001", "3. ETL Process ID")
dbutils.widgets.text("V_ODI_SESS_NO", "999999", "4. ODI Session Number")
dbutils.widgets.text("V_ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "5. Last Extract Time")
dbutils.widgets.text("V_ETL_CURRENT_EXTRACT_TIME", "2024-07-30 12:00:00", "6. Current Extract Time")

## ETL Parameters

In [ ]:
%sql
-- SCEN_TASK_NO {2}: Get ETL last extract time and current extract time
-- Placeholder for ETL parameter table. In a real scenario, this would read from a persistent metadata table.
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp(COALESCE(
    MAX(W_INSERT_DT),
    '${V_ETL_LAST_EXTRACT_TIME}'
), 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time
FROM workspace.test.trg_dep
WHERE DATASOURCE_NUM_ID = ${V_DATASOURCE_NUM_ID};

CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT to_timestamp('${V_ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
print("ETL Parameters:")
print(f"  ETL_JOB_TYPE: {dbutils.widgets.get('V_ETL_JOB_TYPE')}")
print(f"  DATASOURCE_NUM_ID: {dbutils.widgets.get('V_DATASOURCE_NUM_ID')}")
print(f"  ETL_PROC_WID: {dbutils.widgets.get('V_ETL_PROC_WID')}")
print(f"  ODI_SESS_NO: {dbutils.widgets.get('V_ODI_SESS_NO')}")

display(spark.sql("SELECT etl_last_extract_time, etl_current_extract_time FROM v_etl_last_extract_time, v_etl_current_extract_time"))

## Staging Table: `C$_DEP` (`workspace.test.c_dep_stg`)

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Drop staging table
DROP TABLE IF EXISTS workspace.test.c_dep_stg;

In [ ]:
%sql
-- SCEN_TASK_NO {40}: Create staging table C$_DEP
CREATE TABLE workspace.test.c_dep_stg (
    EMPLOYEE_ID    BIGINT,
    FIRST_NAME     STRING,
    LAST_NAME      STRING,
    EMAIL          STRING,
    PHONE_NUMBER   STRING,
    HIRE_DATE      TIMESTAMP,
    JOB_ID         STRING,
    SALARY         DECIMAL(8,2),
    COMMISSION_PCT DECIMAL(2,2),
    MANAGER_ID     BIGINT,
    DEPARTMENT_ID  BIGINT,
    ODI_ROW_ID     STRING
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {50}: Insert into staging table C$_DEP
-- Applying ROW_NUMBER for deduplication based on EMPLOYEE_ID and HIRE_DATE (latest record)
INSERT INTO workspace.test.c_dep_stg
SELECT
    EMPLOYEE_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    HIRE_DATE,
    JOB_ID,
    SALARY,
    COMMISSION_PCT,
    MANAGER_ID,
    DEPARTMENT_ID,
    CAST(monotonically_increasing_id() AS STRING) AS ODI_ROW_ID
FROM (
    SELECT
        src.EMPLOYEE_ID,
        src.FIRST_NAME,
        src.LAST_NAME,
        src.EMAIL,
        src.PHONE_NUMBER,
        src.HIRE_DATE,
        src.JOB_ID,
        src.SALARY,
        src.COMMISSION_PCT,
        src.MANAGER_ID,
        src.DEPARTMENT_ID,
        ROW_NUMBER() OVER (PARTITION BY src.EMPLOYEE_ID ORDER BY src.HIRE_DATE DESC, src.LAST_NAME ASC) AS rn
    FROM workspace.test.departments AS src
    WHERE src.HIRE_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
      AND src.HIRE_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
)
WHERE rn = 1;

In [ ]:
%sql
-- SCEN_TASK_NO {60}: Count records in C$_DEP
SELECT COUNT(*) AS c_dep_stg_record_count FROM workspace.test.c_dep_stg;

## Flow Table: `I$_DEP` (`workspace.test.i_dep_flow`)

In [ ]:
%sql
-- SCEN_TASK_NO {100}: Drop flow table I$_DEP
DROP TABLE IF EXISTS workspace.test.i_dep_flow;

In [ ]:
%sql
-- SCEN_TASK_NO {110}: Create flow table I$_DEP
CREATE TABLE workspace.test.i_dep_flow (
    EMPLOYEE_ID    BIGINT,
    FIRST_NAME     STRING,
    LAST_NAME      STRING,
    EMAIL          STRING,
    PHONE_NUMBER   STRING,
    HIRE_DATE      TIMESTAMP,
    JOB_ID         STRING,
    SALARY         DECIMAL(8,2),
    COMMISSION_PCT DECIMAL(2,2),
    MANAGER_ID     BIGINT,
    DEPARTMENT_ID  BIGINT,
    ODI_ROW_ID     STRING,
    IND_UPDATE     STRING,
    IND_ERROR      STRING,
    ODI_SESS_NO    STRING,
    W_INSERT_DT    TIMESTAMP,
    W_UPDATE_DT    TIMESTAMP
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {120}: Insert into flow table I$_DEP
INSERT INTO workspace.test.i_dep_flow
SELECT
    stg.EMPLOYEE_ID,
    stg.FIRST_NAME,
    stg.LAST_NAME,
    stg.EMAIL,
    stg.PHONE_NUMBER,
    stg.HIRE_DATE,
    stg.JOB_ID,
    stg.SALARY,
    stg.COMMISSION_PCT,
    stg.MANAGER_ID,
    stg.DEPARTMENT_ID,
    stg.ODI_ROW_ID,
    'I' AS IND_UPDATE, -- Initially mark as Insert
    '0' AS IND_ERROR,
    '${V_ODI_SESS_NO}' AS ODI_SESS_NO,
    current_timestamp() AS W_INSERT_DT,
    current_timestamp() AS W_UPDATE_DT
FROM workspace.test.c_dep_stg AS stg;

In [ ]:
%sql
-- SCEN_TASK_NO {130}: Count records in I$_DEP
SELECT COUNT(*) AS i_dep_flow_record_count FROM workspace.test.i_dep_flow;

In [ ]:
%sql
-- SCEN_TASK_NO {140}: Optimize flow table I$_DEP
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.test.i_dep_flow ZORDER BY (EMPLOYEE_ID, DEPARTMENT_ID);

## Error / Check Tables

In [ ]:
%sql
-- SCEN_TASK_NO {150}: Create Error table E$_DEP if not exists
CREATE TABLE IF NOT EXISTS workspace.test.e_dep_err (
    EMPLOYEE_ID    BIGINT,
    FIRST_NAME     STRING,
    LAST_NAME      STRING,
    EMAIL          STRING,
    PHONE_NUMBER   STRING,
    HIRE_DATE      TIMESTAMP,
    JOB_ID         STRING,
    SALARY         DECIMAL(8,2),
    COMMISSION_PCT DECIMAL(2,2),
    MANAGER_ID     BIGINT,
    DEPARTMENT_ID  BIGINT,
    ODI_ROW_ID     STRING,
    ODI_SESS_NO    STRING,
    ERR_CODE       STRING,
    ERR_MESSAGE    STRING,
    ERR_DATE       TIMESTAMP
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {160}: Delete previous session errors from E$_DEP
DELETE FROM workspace.test.e_dep_err
WHERE ODI_SESS_NO = '${V_ODI_SESS_NO}';

In [ ]:
%sql
-- SCEN_TASK_NO {170}: Create ODI check table SNP_CHECK_TAB if not exists
CREATE TABLE IF NOT EXISTS workspace.test.snp_check_tab (
    ODI_SESS_NO      STRING,
    CHECK_NAME       STRING,
    CHECK_TYPE       STRING,
    CHECK_VALUE      BIGINT,
    MESSAGE          STRING,
    INSERT_DATE      TIMESTAMP
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {180}: Delete previous session checks from SNP_CHECK_TAB for this mapping
DELETE FROM workspace.test.snp_check_tab
WHERE ODI_SESS_NO = '${V_ODI_SESS_NO}'
  AND CHECK_NAME = 'PK_DEP';

## PK Violation Detection and Deduplication

In [ ]:
%sql
-- SCEN_TASK_NO {190}: Insert records with Primary Key violations into E$_DEP
INSERT INTO workspace.test.e_dep_err
SELECT
    flow.EMPLOYEE_ID,
    flow.FIRST_NAME,
    flow.LAST_NAME,
    flow.EMAIL,
    flow.PHONE_NUMBER,
    flow.HIRE_DATE,
    flow.JOB_ID,
    flow.SALARY,
    flow.COMMISSION_PCT,
    flow.MANAGER_ID,
    flow.DEPARTMENT_ID,
    flow.ODI_ROW_ID,
    flow.ODI_SESS_NO,
    'PK_VIOLATION' AS ERR_CODE,
    'Duplicate EMPLOYEE_ID found' AS ERR_MESSAGE,
    current_timestamp() AS ERR_DATE
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY EMPLOYEE_ID ORDER BY HIRE_DATE DESC) AS rn
    FROM workspace.test.i_dep_flow
) AS flow
WHERE flow.rn > 1;

In [ ]:
%sql
-- SCEN_TASK_NO {200}: Record PK violation count in SNP_CHECK_TAB
INSERT INTO workspace.test.snp_check_tab
SELECT
    '${V_ODI_SESS_NO}' AS ODI_SESS_NO,
    'PK_DEP' AS CHECK_NAME,
    'COUNT' AS CHECK_TYPE,
    COUNT(*) AS CHECK_VALUE,
    'Number of PK violations for DEP mapping' AS MESSAGE,
    current_timestamp() AS INSERT_DATE
FROM workspace.test.e_dep_err
WHERE ODI_SESS_NO = '${V_ODI_SESS_NO}';

-- SCEN_TASK_NO {210}: Remove duplicates from flow table I$_DEP
CREATE OR REPLACE TEMPORARY VIEW v_deduped_i_dep_flow AS
SELECT
    EMPLOYEE_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    HIRE_DATE,
    JOB_ID,
    SALARY,
    COMMISSION_PCT,
    MANAGER_ID,
    DEPARTMENT_ID,
    ODI_ROW_ID,
    IND_UPDATE,
    IND_ERROR,
    ODI_SESS_NO,
    W_INSERT_DT,
    W_UPDATE_DT
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY EMPLOYEE_ID ORDER BY HIRE_DATE DESC, W_UPDATE_DT DESC) AS rn
    FROM workspace.test.i_dep_flow
)
WHERE rn = 1;

TRUNCATE TABLE workspace.test.i_dep_flow;

INSERT INTO workspace.test.i_dep_flow
SELECT * FROM v_deduped_i_dep_flow;

## Mark Records for Update

In [ ]:
%sql
-- SCEN_TASK_NO {220}: Mark records as 'U' (update) if they already exist in the target
MERGE INTO workspace.test.i_dep_flow AS T
USING (
    SELECT EMPLOYEE_ID
    FROM workspace.test.trg_dep
) AS S
ON T.EMPLOYEE_ID = S.EMPLOYEE_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

## MERGE into Target: `TRG_DEP` (`workspace.test.trg_dep`)

In [ ]:
%sql
-- SCEN_TASK_NO {230}: Create target table TRG_DEP if not exists
CREATE TABLE IF NOT EXISTS workspace.test.trg_dep (
    EMPLOYEE_ID     BIGINT,
    FIRST_NAME      STRING,
    LAST_NAME       STRING,
    EMAIL           STRING,
    PHONE_NUMBER    STRING,
    HIRE_DATE       TIMESTAMP,
    JOB_ID          STRING,
    SALARY          DECIMAL(8,2),
    COMMISSION_PCT  DECIMAL(2,2),
    MANAGER_ID      BIGINT,
    DEPARTMENT_ID   BIGINT,
    W_INSERT_DT     TIMESTAMP,
    W_UPDATE_DT     TIMESTAMP,
    DATASOURCE_NUM_ID BIGINT,
    ETL_PROC_WID    BIGINT
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {240}: Merge flow table I$_DEP into target TRG_DEP
MERGE INTO workspace.test.trg_dep AS T
USING workspace.test.i_dep_flow AS S
ON T.EMPLOYEE_ID = S.EMPLOYEE_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.FIRST_NAME     = S.FIRST_NAME,
    T.LAST_NAME      = S.LAST_NAME,
    T.EMAIL          = S.EMAIL,
    T.PHONE_NUMBER   = S.PHONE_NUMBER,
    T.HIRE_DATE      = S.HIRE_DATE,
    T.JOB_ID         = S.JOB_ID,
    T.SALARY         = S.SALARY,
    T.COMMISSION_PCT = S.COMMISSION_PCT,
    T.MANAGER_ID     = S.MANAGER_ID,
    T.DEPARTMENT_ID  = S.DEPARTMENT_ID,
    T.W_UPDATE_DT    = current_timestamp(),
    T.DATASOURCE_NUM_ID = ${V_DATASOURCE_NUM_ID},
    T.ETL_PROC_WID   = ${V_ETL_PROC_WID}
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    EMPLOYEE_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    HIRE_DATE,
    JOB_ID,
    SALARY,
    COMMISSION_PCT,
    MANAGER_ID,
    DEPARTMENT_ID,
    W_INSERT_DT,
    W_UPDATE_DT,
    DATASOURCE_NUM_ID,
    ETL_PROC_WID
) VALUES (
    S.EMPLOYEE_ID,
    S.FIRST_NAME,
    S.LAST_NAME,
    S.EMAIL,
    S.PHONE_NUMBER,
    S.HIRE_DATE,
    S.JOB_ID,
    S.SALARY,
    S.COMMISSION_PCT,
    S.MANAGER_ID,
    S.DEPARTMENT_ID,
    current_timestamp(),
    current_timestamp(),
    ${V_DATASOURCE_NUM_ID},
    ${V_ETL_PROC_WID}
);

## Optimize Target

In [ ]:
%sql
-- SCEN_TASK_NO {250}: Optimize target table TRG_DEP
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.test.trg_dep ZORDER BY (EMPLOYEE_ID, DEPARTMENT_ID);

## Cleanup

In [ ]:
%sql
-- SCEN_TASK_NO {260}: Drop temporary staging and flow tables
DROP TABLE IF EXISTS workspace.test.c_dep_stg;
DROP TABLE IF EXISTS workspace.test.i_dep_flow;

## Validation

In [ ]:
%sql
-- SCEN_TASK_NO {270}: Final record count for target table
SELECT COUNT(*) AS final_target_record_count FROM workspace.test.trg_dep;

In [ ]:
%sql
-- SCEN_TASK_NO {280}: Sample recently updated records from target
SELECT
    EMPLOYEE_ID,
    FIRST_NAME,
    LAST_NAME,
    HIRE_DATE,
    W_UPDATE_DT,
    W_INSERT_DT
FROM workspace.test.trg_dep
WHERE W_UPDATE_DT >= (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
ORDER BY W_UPDATE_DT DESC
LIMIT 10;

In [ ]:
%sql
-- SCEN_TASK_NO {290}: Error summary from snp_check_tab for current session
SELECT
    CHECK_NAME,
    CHECK_TYPE,
    CHECK_VALUE,
    MESSAGE,
    INSERT_DATE
FROM workspace.test.snp_check_tab
WHERE ODI_SESS_NO = '${V_ODI_SESS_NO}';

## Conversion Notes & Manual Actions Required

1.  **Source Data Type Assumptions**: Since the Oracle DDL was not provided, Spark SQL data types were inferred based on typical usage for employee/department-related columns.
    *   `NUMBER(p,0)` assumed as `BIGINT`.
    *   `NUMBER` with scale assumed as `DECIMAL(p,s)`.
    *   `DATE` assumed as `TIMESTAMP`.
    *   `VARCHAR2` assumed as `STRING`.
2.  **Primary Key (PK) Assumption**: `EMPLOYEE_ID` was assumed as the natural primary key for the `MERGE ON` condition and PK violation detection. This should be verified against the actual Oracle DDL and business rules.
3.  **ETL Parameter Table**: The views `v_etl_last_extract_time` and `v_etl_current_extract_time` currently reference a hypothetical `workspace.test.etl_parameter_table` for `etl_last_extract_time`. This table (or its equivalent source for `MAX(W_INSERT_DT)`) needs to be implemented and populated with the correct metadata if it doesn't already exist. The `V_ETL_LAST_EXTRACT_TIME` widget provides a default fallback.
4.  **`workspace.test.departments` Source Table**: This notebook assumes `workspace.test.departments` exists and contains the source data. Ensure the schema and table are correctly defined and accessible in Databricks.
5.  **Error Handling**: The `E$_DEP` table and PK violation checks (`snp_check_tab`) provide basic error logging. Enhance with more detailed error codes/messages as per business requirements.
6.  **ZORDER Columns**: `(EMPLOYEE_ID, DEPARTMENT_ID)` were selected as ZORDER columns for `OPTIMIZE` statements. Adjust based on query patterns and data access.
7.  **Deduplication**: `ROW_NUMBER()` is used in staging to handle potential duplicate source records, selecting the latest record by `HIRE_DATE`. This replaces the ODI MAX self-join pattern (F.12). Verify if this ordering is appropriate for the business logic.